In [1]:
import json
import pickle
import random
import numpy as np
import nltk
import os

from nltk.stem import WordNetLemmatizer
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD

# custom NLTK download directory on D drive
nltk_data_path = r"D:\download_frameworks\nltk"

# Create folder if not exists
os.makedirs(nltk_data_path, exist_ok=True)

# telling NLTK to use folder
nltk.data.path.append(nltk_data_path)

# Download NLTK data
nltk.download('punkt', download_dir=nltk_data_path)
nltk.download('wordnet', download_dir=nltk_data_path)
nltk.download('omw-1.4', download_dir=nltk_data_path)
nltk.download('punkt_tab', download_dir=nltk_data_path)

print('✅ All libraries imported successfully!')

[nltk_data] Downloading package punkt to
[nltk_data]     D:\download_frameworks\nltk...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     D:\download_frameworks\nltk...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     D:\download_frameworks\nltk...


✅ All libraries imported successfully!


[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     D:\download_frameworks\nltk...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# Load intents.json
with open('intents.json', 'r') as f:
    intents = json.load(f)

print(f'✅ Loaded intents.json')
print(f'Total intent categories: {len(intents["intents"])}')
print('\nIntent tags:')
for intent in intents['intents']:
    print(f'  - {intent["tag"]} ({len(intent["patterns"])} patterns, {len(intent["responses"])} responses)')

✅ Loaded intents.json
Total intent categories: 15

Intent tags:
  - greeting (12 patterns, 4 responses)
  - goodbye (10 patterns, 4 responses)
  - thanks (8 patterns, 4 responses)
  - about (8 patterns, 3 responses)
  - data_science (7 patterns, 3 responses)
  - machine_learning (7 patterns, 3 responses)
  - python (7 patterns, 3 responses)
  - cryptocurrency (8 patterns, 3 responses)
  - crypto_price (7 patterns, 3 responses)
  - time_series (7 patterns, 3 responses)
  - nlp (7 patterns, 3 responses)
  - neural_network (7 patterns, 3 responses)
  - internship (6 patterns, 3 responses)
  - funny (5 patterns, 3 responses)
  - help (7 patterns, 3 responses)


In [3]:
lemmatizer = WordNetLemmatizer()

words    = []   # all unique words from all patterns
classes  = []   # all unique intent tags
documents = []  # (tokenized_words, tag) pairs

# Characters to ignore
ignore_chars = ['?', '!', '.', ',']

for intent in intents['intents']:
    for pattern in intent['patterns']:
        # Tokenize: split sentence into list of words
        word_list = nltk.word_tokenize(pattern)
        words.extend(word_list)
        documents.append((word_list, intent['tag']))

        if intent['tag'] not in classes:
            classes.append(intent['tag'])

# Lemmatize and clean words
words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_chars]
words = sorted(set(words))  # remove duplicates and sort
classes = sorted(set(classes))

# Save for use during prediction
pickle.dump(words,   open('words.pkl',   'wb'))
pickle.dump(classes, open('classes.pkl', 'wb'))

print(f'✅ Preprocessing complete!')
print(f'Total unique words (vocabulary): {len(words)}')
print(f'Total intent classes: {len(classes)}')
print(f'Total training documents: {len(documents)}')
print(f'\nSample words: {words[:15]}')
print(f'Classes: {classes}')

✅ Preprocessing complete!
Total unique words (vocabulary): 147
Total intent classes: 15
Total training documents: 113

Sample words: ["'m", "'s", 'a', 'about', 'activation', 'afternoon', 'algorithm', 'all', 'amdox', 'an', 'analysis', 'analytics', 'answer', 'application', 'appreciated']
Classes: ['about', 'crypto_price', 'cryptocurrency', 'data_science', 'funny', 'goodbye', 'greeting', 'help', 'internship', 'machine_learning', 'neural_network', 'nlp', 'python', 'thanks', 'time_series']


In [4]:
training = []
output_empty = [0] * len(classes)

for document in documents:
    bag = []
    word_patterns = document[0]
    # Lemmatize each word in the pattern
    word_patterns = [lemmatizer.lemmatize(w.lower()) for w in word_patterns]

    # Create bag of words: 1 if word exists in pattern, else 0
    for word in words:
        bag.append(1 if word in word_patterns else 0)

    # Output: 1 for the correct class, 0 for others
    output_row = list(output_empty)
    output_row[classes.index(document[1])] = 1
    training.append([bag, output_row])

# Shuffle and convert to numpy arrays
random.shuffle(training)
training = np.array(training, dtype=object)

X_train = np.array(list(training[:, 0]))  # input features (bag of words)
y_train = np.array(list(training[:, 1]))  # output labels (intent classes)

print(f'✅ Training data created!')
print(f'X_train shape: {X_train.shape}  (samples × vocabulary size)')
print(f'y_train shape: {y_train.shape}  (samples × number of classes)')

✅ Training data created!
X_train shape: (113, 147)  (samples × vocabulary size)
y_train shape: (113, 15)  (samples × number of classes)


In [5]:
# Build the neural network model
model = Sequential()

# Layer 1: 128 neurons, ReLU activation
model.add(Dense(128, input_shape=(len(X_train[0]),), activation='relu'))
model.add(Dropout(0.5))  # 50% dropout to prevent overfitting

# Layer 2: 64 neurons, ReLU activation
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))

# Output layer: softmax gives probability for each class
model.add(Dense(len(y_train[0]), activation='softmax'))

# Compile with SGD optimizer
sgd = SGD(learning_rate=0.01, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

model.summary()

C:\Users\sinha\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 128)                 │          18,944 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 15)                  │             975 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 28,175 (110.06 KB)

 Trainable params: 28,175 (110.06 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import matplotlib.pyplot as plt

# Train the model
history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=5,
    verbose=1
)

# Save the trained model
model.save('chatbot_model.h5')
print('\n✅ Model trained and saved as chatbot_model.h5')

# Plot training accuracy and loss
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(history.history['accuracy'], color='steelblue')
ax1.set_title('Model Accuracy', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'], color='orange')
ax2.set_title('Model Loss', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final Accuracy : {history.history['accuracy'][-1]*100:.2f}%")
print(f"Final Loss     : {history.history['loss'][-1]:.4f}")

Epoch 1/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.0410 - loss: 2.7526
Epoch 2/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0779 - loss: 2.6565
Epoch 3/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2353 - loss: 2.5956
Epoch 4/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1971 - loss: 2.5326
Epoch 5/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2580 - loss: 2.4104
Epoch 6/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3554 - loss: 2.2927
Epoch 7/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.3804 - loss: 2.1548
Epoch 8/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.3707 - loss: 2.0891
Epoch 9/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4728 - loss: 1.8908
Epoch 10/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5350 - loss: 1.7445
Epoch 11/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.4832 - loss: 1.6518
Epoch 12/200
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step

In [ ]:
from tensorflow.keras.models import load_model

# Load saved model and data
model   = load_model('chatbot_model.h5')
words   = pickle.load(open('words.pkl',   'rb'))
classes = pickle.load(open('classes.pkl', 'rb'))

def clean_up_sentence(sentence):
    """Tokenize and lemmatize user input"""
    sentence_words = nltk.word_tokenize(sentence)
    sentence_words = [lemmatizer.lemmatize(w.lower()) for w in sentence_words]
    return sentence_words

def bag_of_words(sentence):
    """Convert sentence to bag of words array"""
    sentence_words = clean_up_sentence(sentence)
    bag = [0] * len(words)
    for w in sentence_words:
        for i, word in enumerate(words):
            if word == w:
                bag[i] = 1
    return np.array(bag)

def predict_class(sentence):
    """Predict the intent class for a given sentence"""
    bow = bag_of_words(sentence)
    res = model.predict(np.array([bow]), verbose=0)[0]

    # Filter out low-confidence predictions
    ERROR_THRESHOLD = 0.25
    results = [[i, r] for i, r in enumerate(res) if r > ERROR_THRESHOLD]
    results.sort(key=lambda x: x[1], reverse=True)  # sort by confidence

    return_list = []
    for r in results:
        return_list.append({
            'intent': classes[r[0]],
            'probability': str(r[1])
        })
    return return_list

def get_response(intents_list, intents_json):
    """Pick a random response for the predicted intent"""
    if not intents_list:
        return "I'm not sure I understand. Could you rephrase that?"

    tag = intents_list[0]['intent']
    for intent in intents_json['intents']:
        if intent['tag'] == tag:
            return random.choice(intent['responses'])

    return "I'm not sure I understand. Could you rephrase that?"

def chatbot_response(text):
    """Main function: takes user input, returns bot response"""
    intents_list = predict_class(text)
    response     = get_response(intents_list, intents)
    return response

print('✅ Chatbot functions ready!')

In [ ]:
# Test the chatbot with sample inputs
test_inputs = [
    "Hello!",
    "What is machine learning?",
    "Tell me about cryptocurrency",
    "What is LSTM?",
    "Tell me a joke",
    "What can you do?",
    "Thanks!",
    "Goodbye"
]

print('=' * 60)
print('           CHATBOT TEST RESPONSES')
print('=' * 60)
for user_input in test_inputs:
    response = chatbot_response(user_input)
    print(f'\n👤 You : {user_input}')
    print(f'🤖 Bot : {response}')
print('\n' + '=' * 60)

In [ ]:
# Interactive chat loop — type 'quit' to exit
print('🤖 Chatbot is ready! Type "quit" to exit.\n')
print('-' * 50)

while True:
    user_input = input('👤 You: ').strip()
    if user_input.lower() in ['quit', 'exit', 'q']:
        print('🤖 Bot: Goodbye! Have a great day!')
        break
    if user_input:
        response = chatbot_response(user_input)
        print(f'🤖 Bot: {response}\n')

In [ ]:
# Save the Streamlit app code
app_code = '''
import streamlit as st
import json
import pickle
import random
import numpy as np
import nltk
from nltk.stem import WordNetLemmatizer
from tensorflow.keras.models import load_model

nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("punkt_tab", quiet=True)

# ── Load model and data ────────────────────────────────────────
@st.cache_resource
def load_chatbot():
    lemmatizer = WordNetLemmatizer()
    model      = load_model("chatbot_model.h5")
    words      = pickle.load(open("words.pkl",   "rb"))
    classes    = pickle.load(open("classes.pkl", "rb"))
    with open("intents.json") as f:
        intents = json.load(f)
    return lemmatizer, model, words, classes, intents

lemmatizer, model, words, classes, intents = load_chatbot()

# ── Chatbot functions ──────────────────────────────────────────
def bag_of_words(sentence):
    sentence_words = nltk.word_tokenize(sentence)
    sentence_words = [lemmatizer.lemmatize(w.lower()) for w in sentence_words]
    bag = [1 if w in sentence_words else 0 for w in words]
    return np.array(bag)

def chatbot_response(text):
    bow = bag_of_words(text)
    res = model.predict(np.array([bow]), verbose=0)[0]
    ERROR_THRESHOLD = 0.25
    results = sorted(
        [[i, r] for i, r in enumerate(res) if r > ERROR_THRESHOLD],
        key=lambda x: x[1], reverse=True
    )
    if not results:
        return "I\'m not sure I understand. Could you rephrase that?"
    tag = classes[results[0][0]]
    for intent in intents["intents"]:
        if intent["tag"] == tag:
            return random.choice(intent["responses"])
    return "I\'m not sure I understand. Could you rephrase that?"

# ── Streamlit UI ───────────────────────────────────────────────
st.set_page_config(page_title="AI Chatbot", page_icon="🤖", layout="centered")

st.title("🤖 Dynamic AI Chatbot")
st.caption("Powered by Python · NLTK · Deep Learning | Amdox Internship Project")
st.markdown("---")

# Initialize chat history
if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "assistant", "content": "Hello! I\'m your AI assistant. Ask me about data science, machine learning, cryptocurrency, or just say hi! 😊"}
    ]

# Display chat messages
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# Chat input
if prompt := st.chat_input("Type your message here..."):
    # Add user message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)

    # Get bot response
    response = chatbot_response(prompt)
    st.session_state.messages.append({"role": "assistant", "content": response})
    with st.chat_message("assistant"):
        st.write(response)

# Sidebar
with st.sidebar:
    st.header("About this Chatbot")
    st.info("""
    **Tech Stack:**
    - Python
    - NLTK (NLP)
    - TensorFlow/Keras
    - Streamlit (UI)

    **Topics I know:**
    - Data Science
    - Machine Learning
    - Deep Learning / NLP
    - Cryptocurrency
    - Time Series
    - Python Libraries
    """)
    if st.button("Clear Chat"):
        st.session_state.messages = [
            {"role": "assistant", "content": "Chat cleared! How can I help you?"}
        ]
        st.rerun()
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code.strip())

print('✅ app.py created!')
print('\nTo run the Streamlit app:')
print('  streamlit run app.py')
print('\nFor Google Colab, use:')
print('  !pip install pyngrok')
print('  from pyngrok import ngrok')
print('  ngrok.set_auth_token("YOUR_TOKEN")')
print('  !streamlit run app.py &')
print('  public_url = ngrok.connect(8501)')
print('  print(public_url)')